In [3]:
# Jalankan ini di sel notebook Anda
!pip install tensorflow scikit-learn nltk pandas

In [4]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split

# Hanya perlu diunduh sekali
nltk.download('stopwords')

# Muat dataset dari file yang Anda unggah
df = pd.read_csv('/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')

# Tampilkan 5 baris pertama untuk diperiksa
print("Data Awal:")
print(df.head())

# Lihat informasi dan tipe data
print("\nInfo Dataset:")
df.info()

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Data Awal:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Info Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [5]:
# Ambil daftar stopwords bahasa Inggris
STOPWORDS = set(stopwords.words('english'))

def clean_text(text):
    """
    Fungsi untuk membersihkan teks:
    1. Hapus tag HTML (<br />)
    2. Hapus karakter selain huruf
    3. Ubah menjadi huruf kecil
    4. Hapus stopwords
    """
    # 1. Hapus tag HTML
    text = re.sub(r'<.*?>', '', text)
    
    # 2. Hapus karakter non-alfabet
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    
    # 3. Ubah ke huruf kecil
    text = text.lower()
    
    # 4. Hapus stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word not in STOPWORDS]
    
    return ' '.join(tokens)

# Terapkan fungsi pembersihan ke kolom 'review'
df['cleaned_review'] = df['review'].apply(clean_text)

print("Data setelah dibersihkan:")
print(df[['review', 'cleaned_review']].head())

Data setelah dibersihkan:
                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                      cleaned_review  
0  one reviewers mentioned watching oz episode ho...  
1  wonderful little production filming technique ...  
2  thought wonderful way spend time hot summer we...  
3  basically family little boy jake thinks zombie...  
4  petter mattei love time money visually stunnin...  


In [6]:
# Mengubah label 'positive' -> 1 dan 'negative' -> 0
df['sentiment_encoded'] = df['sentiment'].apply(lambda x: 1 if x == 'positive' else 0)

print("\nData dengan label numerik:")
print(df[['sentiment', 'sentiment_encoded']].head())


Data dengan label numerik:
  sentiment  sentiment_encoded
0  positive                  1
1  positive                  1
2  positive                  1
3  negative                  0
4  positive                  1


In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Pisahkan data menjadi set pelatihan dan pengujian
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_review'], 
    df['sentiment_encoded'], 
    test_size=0.2, 
    random_state=42
)

# Persiapan Tokenizer
vocab_size = 10000  # Ambil 10.000 kata yang paling umum
oov_tok = "<OOV>"   # Token untuk kata yang tidak ada di vocabulary
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(X_train)

# Ubah teks menjadi sekuens angka
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Padding sekuens agar panjangnya sama
max_length = 200    # Atur panjang maksimum sebuah review
X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length, padding='post', truncating='post')

print("\nUkuran data latih setelah di-padding:", X_train_pad.shape)
print("Ukuran data tes setelah di-padding:", X_test_pad.shape)

2025-07-19 08:45:15.013657: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752914715.198546      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752914715.255329      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered



Ukuran data latih setelah di-padding: (40000, 200)
Ukuran data tes setelah di-padding: (10000, 200)


In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D

embedding_dim = 128

model_lstm = Sequential([
    # 1. Lapisan Embedding: Mengubah indeks kata menjadi vektor padat
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length),
    
    # 2. SpatialDropout1D: Mencegah overfitting dengan mematikan beberapa fitur
    SpatialDropout1D(0.2),
    
    # 3. Lapisan LSTM: Jantung dari model untuk memproses sekuens
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    
    # 4. Lapisan Output: Satu neuron dengan aktivasi sigmoid untuk klasifikasi biner (0 atau 1)
    Dense(1, activation='sigmoid')
])

# Compile model
model_lstm.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Tampilkan ringkasan arsitektur model
model_lstm.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1752914732.390722      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
# Melatih model
num_epochs = 5
batch_size = 64

history = model_lstm.fit(
    X_train_pad, y_train,
    epochs=num_epochs,
    batch_size=batch_size,
    validation_data=(X_test_pad, y_test),
    verbose=1
)

Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 198s 304ms/step - accuracy: 0.5098 - loss: 0.6940 - val_accuracy: 0.5113 - val_loss: 0.6931
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 191s 305ms/step - accuracy: 0.5162 - loss: 0.6906 - val_accuracy: 0.5209 - val_loss: 0.6850
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 189s 303ms/step - accuracy: 0.5609 - loss: 0.6548 - val_accuracy: 0.7837 - val_loss: 0.5240
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 189s 303ms/step - accuracy: 0.8447 - loss: 0.3731 - val_accuracy: 0.8788 - val_loss: 0.2939
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 189s 302ms/step - accuracy: 0.9206 - loss: 0.2120 - val_accuracy: 0.8818 - val_loss: 0.2950


In [10]:
# 1. Evaluasi Akurasi pada data Test
loss, accuracy = model_lstm.evaluate(X_test_pad, y_test)
print(f"\nAkurasi Testing LSTM: {accuracy * 100:.2f}%")

# 2. Cari dan tampilkan kesalahan prediksi
predictions = (model_lstm.predict(X_test_pad) > 0.5).astype("int32").flatten()
y_test_reset = y_test.reset_index(drop=True)
X_test_reset = X_test.reset_index(drop=True)

# Cari indeks di mana prediksi salah
incorrect_indices = y_test_reset[y_test_reset != predictions].index

print("\nContoh Jawaban yang Salah dari Model LSTM:")
print("="*50)
for i, idx in enumerate(incorrect_indices[:5]): # Tampilkan 5 kesalahan pertama
    review_text = X_test_reset[idx]
    true_label = "Positive" if y_test_reset[idx] == 1 else "Negative"
    predicted_label = "Positive" if predictions[idx] == 1 else "Negative"
    
    print(f"Kesalahan #{i+1}")
    print(f"Review: {review_text[:500]}...") # Tampilkan 500 karakter pertama
    print(f"Jawaban Sebenarnya: {true_label}")
    print(f"Prediksi Model: {predicted_label}")
    print("-"*20)

313/313 ━━━━━━━━━━━━━━━━━━━━ 29s 94ms/step - accuracy: 0.8841 - loss: 0.2898

Akurasi Testing LSTM: 88.18%
313/313 ━━━━━━━━━━━━━━━━━━━━ 30s 95ms/step

Contoh Jawaban yang Salah dari Model LSTM:
Kesalahan #1
Review: really liked summerslam due look arena curtains look overall interesting reason anyways could one best summerslam ever wwf lex luger main event yokozuna time ok huge fat man vs strong man glad times changed terrible main event like every match luger terrible matches card razor ramon vs ted dibiase steiner brothers vs heavenly bodies shawn michaels vs curt hening event shawn named big monster body guard diesel irs vs kid bret hart first takes doink takes jerry lawler stuff harts lawler always int...
Jawaban Sebenarnya: Positive
Prediksi Model: Negative
--------------------
Kesalahan #2
Review: okay get purgatory thing first time watched episode seemed like something significant going put finger time costa mesa fires tv really caught attention helped writing essay inferno let 

In [12]:
# Impor library yang dibutuhkan
from transformers import BertTokenizer, TFBertForSequenceClassification
import tensorflow as tf
from sklearn.model_selection import train_test_split
# Anggap df sudah dimuat dan diproses dari langkah sebelumnya

# Pilih model BERT yang akan digunakan
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model_bert = TFBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Gunakan data ASLI (belum dibersihkan) karena tokenizer BERT lebih canggih
X = df['review'].tolist()
y = df['sentiment_encoded'].tolist()

# Bagi data lagi
X_train_bert, X_test_bert, y_train_bert, y_test_bert = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- PERBAIKAN DI SINI ---
# Tokenisasi data untuk BERT tanpa batch_size
train_encodings = tokenizer(X_train_bert, truncation=True, padding=True, max_length=256)
test_encodings = tokenizer(X_test_bert, truncation=True, padding=True, max_length=256)

# Buat dataset TensorFlow (kode ini sudah benar)
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    y_train_bert
)).shuffle(1000).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices((
    dict(test_encodings),
    y_test_bert
)).batch(16)

print("Dataset untuk BERT berhasil dibuat!")
print(train_dataset)

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Dataset untuk BERT berhasil dibuat!
<_BatchDataset element_spec=({'input_ids': TensorSpec(shape=(None, 256), dtype=tf.int32, name=None), 'token_type_ids': TensorSpec(shape=(None, 256), dtype=tf.int32, name=None), 'attention_mask': TensorSpec(shape=(None, 256), dtype=tf.int32, name=None)}, TensorSpec(shape=(None,), dtype=tf.int32, name=None))>


In [20]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.models import Model
from transformers import TFBertModel

# (Pastikan kode tokenizer dan pembuatan dataset `train_dataset`, `test_dataset` 
# dari langkah-langkah sebelumnya sudah dijalankan)

# --- INI ADALAH SOLUSI UTAMA ---

# 1. Buat 'wrapper' Keras Layer untuk TFBertModel
class BertLayer(tf.keras.layers.Layer):
    def __init__(self, bert_model, **kwargs):
        super(BertLayer, self).__init__(**kwargs)
        # Simpan model BERT yang sudah di-load sebelumnya
        self.bert = bert_model

    def call(self, inputs):
        # Saat layer ini dipanggil, 'inputs' adalah dictionary 
        # {'input_ids': tensor, 'attention_mask': tensor}
        # Ini adalah format yang disukai oleh model BERT
        outputs = self.bert(inputs)
        # Kita ambil 'pooler_output' untuk tugas klasifikasi
        return outputs.pooler_output

# 2. Muat model dasar BERT dari Hugging Face
bert_base_model = TFBertModel.from_pretrained('bert-base-uncased')
bert_base_model.trainable = False # Disarankan untuk membekukan bobot agar fine-tuning lebih stabil

# 3. Definisikan input untuk model fungsional kita
input_ids = Input(shape=(256,), dtype=tf.int32, name='input_ids')
attention_mask = Input(shape=(256,), dtype=tf.int32, name='attention_mask')

# 4. Gunakan Custom Layer yang baru kita buat
# Kita berikan dictionary dari KerasTensor sebagai input ke custom layer kita
bert_wrapper_layer = BertLayer(bert_base_model)({
    'input_ids': input_ids, 
    'attention_mask': attention_mask
})

# 5. Bangun sisa model seperti biasa
x = Dropout(0.2)(bert_wrapper_layer)
output_layer = Dense(1, activation='sigmoid', name='output_layer')(x)

# 6. Ciptakan model Keras final
model = Model(inputs=[input_ids, attention_mask], outputs=output_layer)

# 7. Kompilasi model dengan cara standar yang stabil
optimizer = tf.keras.optimizers.Adam(learning_rate=3e-5)
loss = tf.keras.losses.BinaryCrossentropy()
metric = tf.keras.metrics.BinaryAccuracy()

model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

# Tampilkan arsitektur untuk verifikasi
model.summary()

# --- MULAI PELATIHAN ---
print("\nModel berhasil dibuat dengan custom layer. Memulai pelatihan...")
bert_history = model.fit(
    train_dataset,
    epochs=2,
    validation_data=test_dataset
)

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.dense.bias', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ attention_mask      │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_ids           │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bert_layer          │ (None, 768)       │          0 │ attention_mask[0… │
│ (BertLayer)         │                   │            │ input_ids[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 768)       │          0 │ bert_layer[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_layer        │ (None, 1)         │        769 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 769 (3.00 KB)

 Trainable params: 769 (3.00 KB)

...

   1/2500 ━━━━━━━━━━━━━━━━━━━━ 14:25:25 21s/step - binary_accuracy: 0.4375 - loss: 0.7176

I0000 00:00:1752916606.799197     108 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2500/2500 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - binary_accuracy: 0.5300 - loss: 0.7021

W0000 00:00:1752916891.031900     107 assert_op.cc:38] Ignoring Assert operator functional_1_1/bert_layer_1/tf_bert_model_2/bert/embeddings/assert_less/Assert/Assert


2500/2500 ━━━━━━━━━━━━━━━━━━━━ 375s 142ms/step - binary_accuracy: 0.5300 - loss: 0.7021 - val_binary_accuracy: 0.6402 - val_loss: 0.6623
Epoch 2/2
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 348s 139ms/step - binary_accuracy: 0.5670 - loss: 0.6812 - val_binary_accuracy: 0.6736 - val_loss: 0.6417


In [21]:
# 1. Evaluasi akurasi
loss, accuracy = model_bert.evaluate(test_dataset)
print(f"\nAkurasi Testing BERT: {accuracy * 100:.2f}%")

# 2. Dapatkan prediksi untuk analisis kesalahan
predictions_logits = model_bert.predict(test_dataset).logits
bert_predictions = np.argmax(predictions_logits, axis=1)

# Bandingkan dengan jawaban sebenarnya
incorrect_indices_bert = [i for i, (true, pred) in enumerate(zip(y_test_bert, bert_predictions)) if true != pred]

print("\nContoh Jawaban yang Salah dari Model BERT:")
print("="*50)
for i, idx in enumerate(incorrect_indices_bert[:5]):
    review_text = X_test_bert[idx]
    true_label = "Positive" if y_test_bert[idx] == 1 else "Negative"
    predicted_label = "Positive" if bert_predictions[idx] == 1 else "Negative"
    
    print(f"Kesalahan #{i+1}")
    print(f"Review: {review_text[:500]}...")
    print(f"Jawaban Sebenarnya: {true_label}")
    print(f"Prediksi Model: {predicted_label}")
    print("-"*20)


Akurasi Testing BERT: 88.18%
